# 10 -- Otros horizontes de pronóstico (fase C)

Hasta ahora el target era la demanda acumulada a **15 días hábiles**. Aquí se repite la evaluación con
**7 y 30 días** (matrices `matriz_as_of_h7/h30.parquet`, generadas con `07_matriz_as_of` y `HORIZONTE_07=7|30`),
para ver si el horizonte cambia qué tan bien le gana el modelo a la media móvil.

Lo que hay que tener presente al leer los resultados:
- El WAPE absoluto **no es comparable entre horizontes** (un horizonte largo suaviza la demanda y baja el WAPE de
  cualquier método). Lo comparable es la **mejora relativa sobre el piso** del mismo horizonte.
- Los orígenes se espacian `H` días hábiles para no solapar ventanas, así que el número de filas cambia con `H`:
  h=7 aproximadamente duplica las filas y h=30 las reduce a la mitad. La pertenencia a ramas sigue viniendo del
  patrón ADI/CV² diario as-of, no del horizonte.
- Hiperparámetros fijos (los de `MODELOS_POR_RAMA`, afinados a 15 días). Sin búsqueda.

Variantes: `v1` (baseline) y `rel_w` (modelo relativo al nivel, ganador de la fase D en `suave_no_perecedero`).

Cadena: ... -> `07` (h=7, 15, 30) -> `11_modelos_relativos` -> **`10_horizontes`**

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import DW, registrar_huella, verificar_huella, purgar_entrenamiento
import evaluar_nivel1 as ev
import modelos_nivel1 as mn
from exportar_modelos_nivel1 import FEATURES as F1_15, RAMA_A_CONDICION

import json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=Warning, module='sklearn')

pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

EXPERIMENTO = 'EDA-fix-nivel1-10-horizontes'
HORIZONTES = [7, 15, 30]
MODOS = ['v1', 'rel_w']
LLAVES = ['codigo_item', 'sucursal', 'fecha_origen']
B, BOOT_SEED = 2000, 0

def suf(h): return '' if h == 15 else f'_h{h}'
def cols_h(h): return [c.replace('trail_15_prev', f'trail_{h}_prev').replace('trail_15', f'trail_{h}') if c.startswith('trail_15') else c for c in F1_15]

datos = {}
for h in HORIZONTES:
    verificar_huella(f'huella_07{suf(h)}.json', [f'matriz_as_of{suf(h)}.parquet'])
    datos[h] = pd.read_parquet(f'{DW}/matriz_as_of{suf(h)}.parquet')
    assert f'target_demanda_{h}d' in datos[h].columns and all(c in datos[h].columns for c in cols_h(h)), h
print(cols_h(30))
comp = pd.DataFrame({h: {rama: int(cond(datos[h]).sum()) for rama, cond in RAMA_A_CONDICION.items()} for h, _ in [(h, 0) for h in HORIZONTES]})
comp.columns = [f'h={h}' for h in HORIZONTES]
print('\nfilas por rama y horizonte (toda la matriz):')
print(comp)

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 18:41:23 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Huella verificada OK contra huella_07_h7.json.
Huella verificada OK contra huella_07.json.
Huella verificada OK contra huella_07_h30.json.
['trail_30', 'trail_30_prev', 'nivel_medio_60d', 'dias_desde_ultima_venta', 'frecuencia_as_of', 'adi_as_of', 'cv2_as_of', 'racha_max_as_of', 'factor_calendario_ventana', 'cond1_espacio_bodega', 'cond2_perecedero', 'cond3_refrigerado', 'cond4_papel_higienico', 'cond5_temporada']



filas por rama y horizonte (toda la matriz):
                        h=7    h=15    h=30
suave_no_perecedero    5631    2735    1465
suave_perecedero       3714    1800     962
intermitente         534614  257894  136974


## 1. Entrenamiento walk-forward por horizonte y variante

In [2]:
partes, t0 = [], time.time()
for h in HORIZONTES:
    m, cols = datos[h], cols_h(h)
    Y, PISO = f'target_demanda_{h}d', f'pred_media_movil_{h}d'
    for rama, cond in RAMA_A_CONDICION.items():
        dr = m.loc[cond(m)]
        for f in [2, 3, 4, 5]:
            tr, te = purgar_entrenamiento(dr, f, horizonte=h, verbose=False), dr[dr['fold_id'] == f]   # sin ventanas solapadas con el test
            if len(tr) < ev.MIN_FILAS_ENTRENAMIENTO or len(te) == 0:
                print(f'h={h} {rama} fold {f}: sin datos suficientes (train={len(tr)}, test={len(te)})'); continue
            blk = te[LLAVES + ['fold_id']].copy()
            blk['y'], blk['piso'], blk['rama'], blk['h'] = te[Y].to_numpy(), te[PISO].to_numpy(), rama, h
            for modo in MODOS:
                blk[f'pred_{modo}'] = mn.predecir_q50(rama, modo, tr, te, cols, Y, PISO, h)
            partes.append(blk)
            print(f'h={h:>2} {rama:22s} fold {f}: train={len(tr):>7,} test={len(te):>6,} ({time.time()-t0:,.0f}s)')
res = pd.concat(partes, ignore_index=True)
print(res.shape)

h= 7 suave_no_perecedero    fold 2: train=    585 test= 1,742 (2s)


h= 7 suave_no_perecedero    fold 3: train=  2,325 test=   567 (5s)


h= 7 suave_no_perecedero    fold 4: train=  2,896 test= 1,320 (7s)


h= 7 suave_no_perecedero    fold 5: train=  4,159 test= 1,352 (9s)


h= 7 suave_perecedero       fold 2: train=    378 test= 1,092 (10s)


...

h=30 suave_perecedero       fold 5: train=    700 test=   222 (261s)


h=30 intermitente           fold 2: train= 10,076 test=36,722 (266s)


h=30 intermitente           fold 3: train= 46,590 test=17,310 (279s)


h=30 intermitente           fold 4: train= 63,376 test=29,800 (295s)


h=30 intermitente           fold 5: train= 92,986 test=38,028 (316s)
(851870, 10)


## 2. Controles de replicación (h=15)

Con el horizonte por defecto, `v1` debe reproducir 08 y `rel_w` debe reproducir la fase D (11). Si no, el módulo
`modelos_nivel1.py` o el parametrizado de `07` cambiaron algo.

In [3]:
r15 = res[res['h'] == 15]
d08 = pd.read_parquet(f'{DW}/evaluacion_nivel_1.parquet')[LLAVES + ['rama', 'pred_q50']]
c8 = r15.merge(d08, on=LLAVES + ['rama'], validate='one_to_one')
assert len(c8) == len(r15)
dif08 = float((c8['pred_q50'] - c8['pred_v1']).abs().max())
d11 = pd.read_parquet(f'{DW}/preds_modelos_relativos_11.parquet')[LLAVES + ['rama', 'pred_rel_w']].rename(columns={'pred_rel_w': 'pred_11'})
c11 = r15.merge(d11, on=LLAVES + ['rama'], validate='one_to_one')
assert len(c11) == len(r15)
dif11 = float((c11['pred_11'] - c11['pred_rel_w']).abs().max())
print(f'h=15: dif máxima v1 vs 08 = {dif08:.6f} | rel_w vs 11 = {dif11:.6f}')
assert dif08 < 1e-3 and dif11 < 1e-3, 'no reproduce 08 / 11'
print('replicación OK')

h=15: dif máxima v1 vs 08 = 0.000000 | rel_w vs 11 = 0.000000
replicación OK


## 3. Mejora sobre el piso por horizonte, rama y variante (IC95, bootstrap de productos)

In [4]:
filas = []
for (h, rama), g in res.groupby(['h', 'rama']):
    for modo in MODOS:
        col = f'pred_{modo}'
        p = ev.mejora_con_ic(g, 'y', col, 'piso', B, BOOT_SEED)
        s = ev.mejora_sin_peor_producto(g, 'y', col, 'piso', B, BOOT_SEED)
        fila = dict(h=h, rama=rama, modo=modo, n_filas=p['n_filas'], n_productos=p['n_productos'], wape=p['wape_modelo'],
                    wape_piso=p['wape_piso'], mejora_vs_piso=p['mejora_rel'], ic_bajo=p['ic_bajo'], ic_alto=p['ic_alto'],
                    supera_con_ic=bool(p['ic_bajo'] > 0), sin_peor=s['producto_excluido'],
                    mejora_sin_peor=s['mejora_rel'], sin_peor_ic_bajo=s['ic_bajo'], sin_peor_ic_alto=s['ic_alto'])
        if modo != 'v1':
            q = ev.mejora_con_ic(g, 'y', col, 'pred_v1', B, BOOT_SEED)
            h5 = ev.mejora_con_ic(g[g['fold_id'] == ev.FOLD_HOLDOUT], 'y', col, 'pred_v1', B, BOOT_SEED)
            b24 = ev.mejora_con_ic(g[g['fold_id'].isin(ev.FOLDS_BUSQUEDA)], 'y', col, 'pred_v1', B, BOOT_SEED)
            fila.update(mejora_vs_v1=q['mejora_rel'], v1_ic_bajo=q['ic_bajo'], v1_ic_alto=q['ic_alto'],
                        mejora_vs_v1_f2a4=b24['mejora_rel'], f2a4_ic_bajo=b24['ic_bajo'], f2a4_ic_alto=b24['ic_alto'],
                        mejora_vs_v1_f5=h5['mejora_rel'])
        filas.append(fila)
tabla = pd.DataFrame(filas)
for rama in tabla['rama'].unique():
    print(f'\n=== {rama} ===')
    print(tabla[tabla['rama'] == rama][['h', 'modo', 'n_filas', 'n_productos', 'wape', 'wape_piso', 'mejora_vs_piso', 'ic_bajo',
                                          'ic_alto', 'supera_con_ic', 'mejora_sin_peor', 'sin_peor_ic_bajo']].to_string(index=False))


=== intermitente ===
 h  modo  n_filas  n_productos  wape  wape_piso  mejora_vs_piso  ic_bajo  ic_alto  supera_con_ic  mejora_sin_peor  sin_peor_ic_bajo
 7    v1   484234         3050 0.532      0.559           0.047    0.002    0.078           True            0.071             0.061
 7 rel_w   484234         3050 0.517      0.559           0.075    0.069    0.082           True            0.076             0.070
15    v1   232704         3050 0.433      0.438           0.012   -0.061    0.056          False            0.049             0.040
15 rel_w   232704         3050 0.422      0.438           0.036    0.008    0.054           True            0.049             0.042
30    v1   121860         3050 0.373      0.365          -0.021   -0.148    0.057          False            0.043             0.023
30 rel_w   121860         3050 0.350      0.365           0.043    0.029    0.057           True            0.047             0.035

=== suave_no_perecedero ===
 h  modo  n_filas  n_prod

## 4. Decisión mecánica

Por rama y horizonte: ¿algún modelo supera el piso con IC95 > 0? Y para `rel_w` contra `v1`, lo mismo que en 07c/11
(se elige con los folds 2-4, se confirma en el fold 5).

In [5]:
decision = {'notebook': '10_horizontes', 'fecha_generacion': pd.Timestamp.now().isoformat(),
            'politica_holdout': ev.POLITICA_HOLDOUT, 'hiperparametros': 'fijos (afinados a 15 dias, sin busqueda)',
            'replicacion': {'dif_vs_08': dif08, 'dif_vs_11': dif11}, 'horizontes': {}}
for h in HORIZONTES:
    decision['horizontes'][str(h)] = {}
    for rama in tabla['rama'].unique():
        t = tabla[(tabla['h'] == h) & (tabla['rama'] == rama)].set_index('modo')
        rw, v1 = t.loc['rel_w'], t.loc['v1']
        adopta_rel = bool(rw['f2a4_ic_bajo'] > 0 and rw['mejora_vs_v1_f5'] > 0)
        decision['horizontes'][str(h)][rama] = dict(
            n_filas=int(rw['n_filas']), n_productos=int(rw['n_productos']),
            v1_mejora_vs_piso=float(v1['mejora_vs_piso']), v1_supera_con_ic=bool(v1['supera_con_ic']),
            rel_w_mejora_vs_piso=float(rw['mejora_vs_piso']), rel_w_ic=[float(rw['ic_bajo']), float(rw['ic_alto'])],
            rel_w_supera_con_ic=bool(rw['supera_con_ic']), rel_w_vs_v1_f2a4=float(rw['mejora_vs_v1_f2a4']),
            rel_w_vs_v1_f2a4_ic=[float(rw['f2a4_ic_bajo']), float(rw['f2a4_ic_alto'])],
            rel_w_vs_v1_f5=float(rw['mejora_vs_v1_f5']), rel_w_adopta_sobre_v1=adopta_rel)
        print(f"h={h:>2} {rama:22s} v1 {v1['mejora_vs_piso']:+.1%} {'(IC>0)' if v1['supera_con_ic'] else ''}| "
              f"rel_w {rw['mejora_vs_piso']:+.1%} IC [{rw['ic_bajo']:+.1%}, {rw['ic_alto']:+.1%}] "
              f"{'SUPERA el piso' if rw['supera_con_ic'] else 'no supera'} | rel_w vs v1 (2-4) {rw['mejora_vs_v1_f2a4']:+.1%}, f5 {rw['mejora_vs_v1_f5']:+.1%}")
with open(f'{DW}/decision_10.json', 'w') as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)
tabla.to_csv(f'{DW}/resultados_horizontes_10.csv', index=False)
res.to_parquet(f'{DW}/preds_horizontes_10.parquet', index=False)
huella = {'notebook': '10_horizontes', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'entradas': registrar_huella(['matriz_as_of.parquet', 'matriz_as_of_h7.parquet', 'matriz_as_of_h30.parquet',
                                        'evaluacion_nivel_1.parquet', 'preds_modelos_relativos_11.parquet']),
          'salidas': registrar_huella(['decision_10.json', 'resultados_horizontes_10.csv', 'preds_horizontes_10.parquet'])}
with open(f'{DW}/huella_10.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
print('guardado: decision_10.json, resultados_horizontes_10.csv, preds_horizontes_10.parquet, huella_10.json')

h= 7 intermitente           v1 +4.7% (IC>0)| rel_w +7.5% IC [+6.9%, +8.2%] SUPERA el piso | rel_w vs v1 (2-4) +3.5%, f5 +1.7%
h= 7 suave_no_perecedero    v1 +6.3% (IC>0)| rel_w +10.0% IC [+6.3%, +15.0%] SUPERA el piso | rel_w vs v1 (2-4) +3.7%, f5 +4.4%
h= 7 suave_perecedero       v1 -4.2% | rel_w -2.1% IC [-3.4%, +1.1%] no supera | rel_w vs v1 (2-4) +1.7%, f5 +3.1%
h=15 intermitente           v1 +1.2% | rel_w +3.6% IC [+0.8%, +5.4%] SUPERA el piso | rel_w vs v1 (2-4) +3.0%, f5 +1.1%
h=15 suave_no_perecedero    v1 +1.0% | rel_w +2.2% IC [-1.8%, +7.0%] no supera | rel_w vs v1 (2-4) +2.8%, f5 -4.1%
h=15 suave_perecedero       v1 -2.5% | rel_w -8.0% IC [-11.3%, +1.9%] no supera | rel_w vs v1 (2-4) -2.1%, f5 -11.2%
h=30 intermitente           v1 -2.1% | rel_w +4.3% IC [+2.9%, +5.7%] SUPERA el piso | rel_w vs v1 (2-4) +8.4%, f5 -0.1%
h=30 suave_no_perecedero    v1 -4.4% | rel_w +0.1% IC [-4.3%, +4.3%] no supera | rel_w vs v1 (2-4) -0.3%, f5 +12.4%
h=30 suave_perecedero       v1 -25.5% | rel

guardado: decision_10.json, resultados_horizontes_10.csv, preds_horizontes_10.parquet, huella_10.json


## 5. Registro en MLflow

In [6]:
for _, r in tabla.iterrows():
    metrics = {k: r[k] for k in ['n_filas', 'n_productos', 'wape', 'wape_piso', 'mejora_vs_piso', 'ic_bajo', 'ic_alto',
                                 'mejora_sin_peor', 'sin_peor_ic_bajo', 'sin_peor_ic_alto']}
    metrics['supera_con_ic'] = int(r['supera_con_ic'])
    if r['modo'] != 'v1':
        metrics.update({k: r[k] for k in ['mejora_vs_v1', 'v1_ic_bajo', 'v1_ic_alto', 'mejora_vs_v1_f2a4', 'f2a4_ic_bajo',
                                          'f2a4_ic_alto', 'mejora_vs_v1_f5']})
    ev.registrar_mlflow(EXPERIMENTO, f"h{int(r['h'])}__{r['rama']}__{r['modo']}",
        tags={'fase': 'C', 'notebook': '10', 'horizonte': int(r['h']), 'rama': r['rama'], 'modo': r['modo'],
              'rama_git': 'fix/eda-modelado-nivel1'},
        params={'horizonte_dias_habiles': int(r['h']), 'features': ','.join(cols_h(int(r['h']))),
                'hiperparametros': 'fijos MODELOS_POR_RAMA (afinados a 15d)', 'bootstrap_B': B},
        metrics=metrics, artefactos=[f'{DW}/decision_10.json'] if (r['h'] == 30 and r['modo'] == 'rel_w') else ())
print(f'{len(tabla)} runs registrados en {EXPERIMENTO}')

🏃 View run h7__intermitente__v1 at: http://127.0.0.1:5000/#/experiments/18/runs/d38eaec305b2401c9091bcd731c7d768
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h7__intermitente__rel_w at: http://127.0.0.1:5000/#/experiments/18/runs/5c30516659ec4ba788a131d05cc9539b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h7__suave_no_perecedero__v1 at: http://127.0.0.1:5000/#/experiments/18/runs/ec03533299d346d2904183f20e07e590
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h7__suave_no_perecedero__rel_w at: http://127.0.0.1:5000/#/experiments/18/runs/71443929026942f68d95cd170e4e2f78
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h7__suave_perecedero__v1 at: http://127.0.0.1:5000/#/experiments/18/runs/00bf834b153d43bd9461ec73c1acd8f5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


...

🏃 View run h30__intermitente__rel_w at: http://127.0.0.1:5000/#/experiments/18/runs/a813bd56abb7447c948d0e14c1230f18
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h30__suave_no_perecedero__v1 at: http://127.0.0.1:5000/#/experiments/18/runs/45bc9837b35b4eeeba7b406bdb147c2c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h30__suave_no_perecedero__rel_w at: http://127.0.0.1:5000/#/experiments/18/runs/09311cd849d74a95b7753905ab291f91
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h30__suave_perecedero__v1 at: http://127.0.0.1:5000/#/experiments/18/runs/c36ffd11ca7c41a99d934b23d416ac6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18


🏃 View run h30__suave_perecedero__rel_w at: http://127.0.0.1:5000/#/experiments/18/runs/fb19e918ffda4789a6c8c00d85d6f178
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/18
18 runs registrados en EDA-fix-nivel1-10-horizontes
